In [ ]:
# ============================================================
# AI REALIZATION GAP — EU27
# STEP 07 — ECONOMETRIC MODELS
# ============================================================

from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI_Realization_Gap_EU27"
)

PROCESSED_DIR = PROJECT_ROOT / "03_processed_data"
TABLE_DIR = PROJECT_ROOT / "05_tables"
MODEL_DIR = PROJECT_ROOT / "07_models"
LOG_DIR = PROJECT_ROOT / "08_logs"

for folder in [
    TABLE_DIR,
    MODEL_DIR,
    LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

Mounted at /content/drive
/content/drive/MyDrive/AI_Realization_Gap_EU27


In [ ]:
# ============================================================
# INSTALL PANEL MODEL PACKAGE
# ============================================================

!pip -q install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.7 MB/s eta 0:00:00


In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
# ============================================================
# LOAD MASTER DATASET
# ============================================================

master_path = (
    PROCESSED_DIR /
    "Step_05_Master_AI_Realization_Gap_EU27.csv"
)

df = pd.read_csv(master_path)

print("Shape:", df.shape)
print("Countries:", df["iso3"].nunique())
print("Years:", sorted(df["year"].unique()))

display(df.head())

Shape: (108, 22)
Countries: 27
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]


,geo,country,iso3,year,ai_adoption_pct,ai_adoption,ai_adoption_norm,z_ai_adoption,ict_specialists_pct,hrst_pct,...,ai_potential,ai_potential_equal_z,ai_potential_equal,ai_realization_gap,ai_realization_gap_z,ai_realization_gap_equal,vhcn_pct,gdp_pc_pps,ln_gdp_pc_pps,cloud_pct
0,AT,Austria,AUT,2021,8.83,0.0883,0.183272,-0.448175,4.5,52.3,...,0.412078,-0.365939,0.401139,0.228807,0.094730,0.217868,45.4,40185.6,10.601264,39.73
1,AT,Austria,AUT,2023,10.79,0.1079,0.231488,-0.223323,5.3,54.5,...,0.520042,0.047932,0.507658,0.288554,0.283173,0.276170,67.6,46391.9,10.744880,46.06
2,AT,Austria,AUT,2024,20.27,0.2027,0.464699,0.864228,5.3,55.2,...,0.532790,0.097992,0.520542,0.068092,-0.755577,0.055844,72.2,47527.9,10.769072,NaN
3,AT,Austria,AUT,2025,29.95,0.2995,0.702829,1.974723,5.4,55.3,...,0.552239,0.175174,0.540407,-0.150590,-1.791622,-0.162422,76.3,49029.2,10.800171,51.68
4,BE,Belgium,BEL,2021,10.32,0.1032,0.219926,-0.277241,5.6,58.7,...,0.615464,0.417378,0.602743,0.395538,0.702375,0.382817,68.9,38699.2,10.563574,52.50


In [ ]:
# ============================================================
# DATA VALIDATION — fail early if input is incomplete
# ============================================================
EXPECTED_YEARS = {2021, 2023, 2024, 2025}
REQUIRED_COLUMNS = ["iso3", "year", "ai_realization_gap", "vhcn_pct", "ln_gdp_pc_pps"]
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing Step 05 columns: {missing_columns}")
if df.duplicated(["iso3", "year"]).any():
    raise ValueError("Duplicate iso3-year observations")
if len(df) != 108 or df["iso3"].nunique() != 27 or set(df["year"]) != EXPECTED_YEARS:
    raise ValueError("Expected balanced EU-27 data: 27 countries x 4 years (108 rows)")
if df[REQUIRED_COLUMNS].isna().any().any():
    raise ValueError("Missing values in model variables")
if not df.groupby("iso3")["year"].nunique().eq(4).all():
    raise ValueError("Unbalanced country-year panel")
print("PASS: EU-27, four waves, 108 observations, no missing model variables")


In [ ]:
# ============================================================
# 0. PANEL STRUCTURE
# ============================================================

print("Observations:", len(df))
print("Countries (N):", df["iso3"].nunique())
print("Time periods (T):", df["year"].nunique())

counts = (
    df.groupby("iso3")
    .size()
)

print("\nObservations per country:")
print(counts.value_counts().sort_index())

balanced = counts.nunique() == 1

print("\nBalanced panel:", balanced)

Observations: 108
Countries (N): 27
Time periods (T): 4

Observations per country:
4    27
Name: count, dtype: int64

Balanced panel: True


In [ ]:
!pip -q install linearmodels

from linearmodels.panel import (
    PanelOLS,
    RandomEffects,
    PooledOLS
)

In [ ]:
panel = (
    df
    .set_index(["iso3", "year"])
    .sort_index()
)

In [ ]:
panel["const"] = 1.0

In [ ]:
y = panel["ai_realization_gap"]

X = panel[
    [
        "const",
        "vhcn_pct",
        "ln_gdp_pc_pps"
    ]
]

pooled_model = PooledOLS(
    y,
    X
)

pooled_res = pooled_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(pooled_res)

                          PooledOLS Estimation Summary                          
Dep. Variable:     ai_realization_gap   R-squared:                        0.0656
Estimator:                  PooledOLS   R-squared (Between):              0.1531
No. Observations:                 108   R-squared (Within):              -0.0620
Date:                Thu, Oct 01 2026   R-squared (Overall):              0.0656
Time:                        08:51:39   Log-likelihood                    21.373
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      3.6833
Entities:                          27   P-value                           0.0284
Avg Obs:                       4.0000   Distribution:                   F(2,105)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             0.9772
                            

In [ ]:
fe_model = PanelOLS(
    y,
    panel[
        [
            "vhcn_pct",
            "ln_gdp_pc_pps"
        ]
    ],
    entity_effects=True
)

fe_res = fe_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(fe_res)

                          PanelOLS Estimation Summary                           
Dep. Variable:     ai_realization_gap   R-squared:                        0.1969
Estimator:                   PanelOLS   R-squared (Between):             -639.30
No. Observations:                 108   R-squared (Within):               0.1969
Date:                Thu, Oct 01 2026   R-squared (Overall):             -521.03
Time:                        08:51:51   Log-likelihood                    78.093
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      9.6828
Entities:                          27   P-value                           0.0002
Avg Obs:                       4.0000   Distribution:                    F(2,79)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             7.5116
                            

In [ ]:
re_model = RandomEffects(
    y,
    X
)

re_res = re_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(re_res)

                        RandomEffects Estimation Summary                        
Dep. Variable:     ai_realization_gap   R-squared:                        0.0344
Estimator:              RandomEffects   R-squared (Between):             -0.0540
No. Observations:                 108   R-squared (Within):               0.0608
Date:                Thu, Oct 01 2026   R-squared (Overall):             -0.0073
Time:                        08:52:03   Log-likelihood                    54.028
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1.8685
Entities:                          27   P-value                           0.1595
Avg Obs:                       4.0000   Distribution:                   F(2,105)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             1.7166
                            

In [ ]:
import statsmodels.formula.api as smf

pooled_sm = smf.ols(
    "ai_realization_gap ~ vhcn_pct + ln_gdp_pc_pps",
    data=df
).fit()

fe_sm = smf.ols(
    "ai_realization_gap ~ vhcn_pct + ln_gdp_pc_pps + C(iso3)",
    data=df
).fit()

f_test = fe_sm.compare_f_test(
    pooled_sm
)

print("F statistic:", f_test[0])
print("p-value:", f_test[1])
print("df difference:", f_test[2])

F statistic: 5.647768833928346
p-value: 1.173255951365251e-09
df difference: 26.0


In [ ]:
# ============================================================
# BREUSCH-PAGAN LM FOR RANDOM EFFECTS
# ============================================================

pooled_plain = smf.ols(
    "ai_realization_gap ~ vhcn_pct + ln_gdp_pc_pps",
    data=df
).fit()

df_lm = df.copy()

df_lm["resid"] = pooled_plain.resid

N = df_lm["iso3"].nunique()
T = df_lm["year"].nunique()

sum_resid_by_country = (
    df_lm.groupby("iso3")["resid"].sum()
)

numerator = (
    sum_resid_by_country.pow(2).sum()
)

denominator = (
    (df_lm["resid"] ** 2).sum()
)

LM = (
    N * T / (2 * (T - 1))
) * (
    (
        numerator / denominator
    ) - 1
) ** 2

from scipy.stats import chi2

p_lm = 1 - chi2.cdf(
    LM,
    df=1
)

print("Breusch-Pagan LM:", LM)
print("p-value:", p_lm)

Breusch-Pagan LM: 23.79627026529829
p-value: 1.0708976991846697e-06


In [ ]:
# ============================================================
# MUNDLAK DEVICE
# ============================================================

df["mean_vhcn"] = (
    df.groupby("iso3")[
        "vhcn_pct"
    ].transform("mean")
)

df["mean_ln_gdp"] = (
    df.groupby("iso3")[
        "ln_gdp_pc_pps"
    ].transform("mean")
)

In [ ]:
mundlak = smf.ols(
    """
    ai_realization_gap ~
    vhcn_pct +
    ln_gdp_pc_pps +
    mean_vhcn +
    mean_ln_gdp +
    C(year)
    """,
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": df["iso3"]
    }
)

print(mundlak.summary())

                            OLS Regression Results                            
Dep. Variable:     ai_realization_gap   R-squared:                       0.383
Model:                            OLS   Adj. R-squared:                  0.340
Method:                 Least Squares   F-statistic:                     23.29
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           1.13e-09
Time:                        08:53:39   Log-Likelihood:                 43.777
No. Observations:                 108   AIC:                            -71.55
Df Residuals:                     100   BIC:                            -50.10
Df Model:                           7                                         
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -1.7252      1.252     

In [ ]:
mundlak_test = mundlak.f_test(
    "mean_vhcn = 0, mean_ln_gdp = 0"
)

print(mundlak_test)

<F test: F=8.327815426317347, p=0.001603100400448638, df_denom=26, df_num=2>


In [ ]:
# ============================================================
# TWO-WAY FIXED EFFECTS
# ============================================================

twfe_model = PanelOLS(
    dependent=panel[
        "ai_realization_gap"
    ],

    exog=panel[
        [
            "vhcn_pct",
            "ln_gdp_pc_pps"
        ]
    ],

    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)

twfe_res = twfe_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(twfe_res)

                          PanelOLS Estimation Summary                           
Dep. Variable:     ai_realization_gap   R-squared:                        0.1919
Estimator:                   PanelOLS   R-squared (Between):             -2939.7
No. Observations:                 108   R-squared (Within):              -2.0640
Date:                Thu, Oct 01 2026   R-squared (Overall):             -2396.4
Time:                        08:54:15   Log-likelihood                    126.75
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      9.0236
Entities:                          27   P-value                           0.0003
Avg Obs:                       4.0000   Distribution:                    F(2,76)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             10.360
                            

In [ ]:
df_diag = df.copy()

fe_diag = smf.ols(
    """
    ai_realization_gap ~
    vhcn_pct +
    ln_gdp_pc_pps +
    C(iso3) +
    C(year)
    """,
    data=df_diag
).fit()

df_diag["resid"] = fe_diag.resid

In [ ]:
country_resid_var = (
    df_diag
    .groupby("iso3")["resid"]
    .var()
    .sort_values()
)

display(
    country_resid_var
)

,resid
iso3,
DEU,0.000476
IRL,0.000600
MLT,0.000604
ITA,0.000703
CZE,0.000908
FRA,0.001369
SVK,0.002541
ESP,0.002777
CYP,0.002898


In [ ]:
# ============================================================
# SERIAL CORRELATION DIAGNOSTIC
# ============================================================

diag = df_diag.sort_values(
    ["iso3", "year"]
).copy()

diag["resid_lag"] = (
    diag.groupby("iso3")[
        "resid"
    ].shift(1)
)

diag_serial = diag.dropna(
    subset=["resid_lag"]
)

serial_model = smf.ols(
    "resid ~ resid_lag",
    data=diag_serial
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
        diag_serial["iso3"]
    }
)

print(serial_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  resid   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.009
Method:                 Least Squares   F-statistic:                    0.8115
Date:                Thu, 01 Oct 2026   Prob (F-statistic):              0.376
Time:                        08:56:50   Log-Likelihood:                 95.138
No. Observations:                  81   AIC:                            -186.3
Df Residuals:                      79   BIC:                            -181.5
Df Model:                           1                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    9.79e-15      0.005   2.02e-12      1.0

In [ ]:
# ============================================================
# PESARAN CD ON RESIDUALS
# ============================================================

resid_panel = (
    df_diag
    .pivot(
        index="year",
        columns="iso3",
        values="resid"
    )
)

corr_matrix = (
    resid_panel.corr()
)

N = corr_matrix.shape[0]
T = resid_panel.shape[0]

upper_corr = []

for i in range(N):
    for j in range(i + 1, N):

        rho = corr_matrix.iloc[i, j]

        if np.isfinite(rho):
            upper_corr.append(rho)

upper_corr = np.array(
    upper_corr
)

CD = np.sqrt(
    2 * T /
    (N * (N - 1))
) * upper_corr.sum()

from scipy.stats import norm

p_cd = 2 * (
    1 - norm.cdf(abs(CD))
)

print("Pesaran CD:", CD)
print("p-value:", p_cd)

print(
    "Mean pairwise residual correlation:",
    upper_corr.mean()
)

Pesaran CD: -1.1866699948512989
p-value: 0.23535781484645102
Mean pairwise residual correlation: -0.031669879241905126


In [ ]:
# ============================================================
# STEP 07 — SAVE ALL ECONOMETRIC RESULTS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
import shutil

STEP07_DIR = TABLE_DIR / "Step_07_All_Results"
STEP07_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:")
print(STEP07_DIR)


# ============================================================
# 1. PANEL STRUCTURE
# ============================================================

country_counts = df.groupby("iso3").size()

panel_structure = pd.DataFrame({
    "statistic": [
        "Number of observations",
        "Number of countries (N)",
        "Number of time periods (T)",
        "Minimum observations per country",
        "Maximum observations per country",
        "Balanced panel"
    ],
    "value": [
        len(df),
        df["iso3"].nunique(),
        df["year"].nunique(),
        country_counts.min(),
        country_counts.max(),
        country_counts.nunique() == 1
    ]
})

panel_structure.to_csv(
    STEP07_DIR / "Step_07_01_Panel_Structure.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 2. DESCRIPTIVE STATISTICS — MODEL VARIABLES
# ============================================================

desc_vars = [
    "ai_realization_gap",
    "vhcn_pct",
    "gdp_pc_pps",
    "ln_gdp_pc_pps"
]

# include robustness variables if they exist
for v in [
    "ai_realization_gap_z",
    "ai_realization_gap_equal",
    "cloud_pct"
]:
    if v in df.columns:
        desc_vars.append(v)

descriptive = (
    df[desc_vars]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

descriptive.to_csv(
    STEP07_DIR / "Step_07_02_Descriptive_Statistics.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 3. POOLED OLS VS FIXED EFFECTS F TEST
# ============================================================

# f_test from:
# f_test = fe_sm.compare_f_test(pooled_sm)

f_selection = pd.DataFrame({
    "test": [
        "Joint significance of country fixed effects"
    ],
    "null_hypothesis": [
        "Country fixed effects are jointly zero; pooled OLS is sufficient"
    ],
    "statistic": [
        float(f_test[0])
    ],
    "p_value": [
        float(f_test[1])
    ],
    "df_difference": [
        float(f_test[2])
    ],
    "decision_5pct": [
        "Reject pooled OLS" if f_test[1] < 0.05
        else "Do not reject pooled OLS"
    ]
})

f_selection.to_csv(
    STEP07_DIR / "Step_07_03_Pooled_vs_FE_F_Test.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 4. MUNDLAK TEST
# ============================================================

def scalar_value(x):
    """Safely convert test output to scalar."""
    try:
        arr = np.asarray(x).squeeze()
        return float(arr)
    except:
        return np.nan


mundlak_test_table = pd.DataFrame({
    "test": [
        "Mundlak joint significance test"
    ],
    "null_hypothesis": [
        "Country means of time-varying regressors are jointly zero"
    ],
    "F_statistic": [
        scalar_value(mundlak_test.fvalue)
    ],
    "p_value": [
        scalar_value(mundlak_test.pvalue)
    ],
    "df_num": [
        scalar_value(getattr(mundlak_test, "df_num", np.nan))
    ],
    "df_denom": [
        scalar_value(getattr(mundlak_test, "df_denom", np.nan))
    ],
    "decision_5pct": [
        "Reject RE exogeneity assumption"
        if scalar_value(mundlak_test.pvalue) < 0.05
        else "Do not reject RE exogeneity assumption"
    ]
})

mundlak_test_table.to_csv(
    STEP07_DIR / "Step_07_04_Mundlak_Test.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 5. MUNDLAK MODEL COEFFICIENTS
# ============================================================

mundlak_ci = mundlak.conf_int()

mundlak_coef = pd.DataFrame({
    "variable": mundlak.params.index,
    "coefficient": mundlak.params.values,
    "std_error": mundlak.bse.values,
    "test_statistic": mundlak.tvalues.values,
    "p_value": mundlak.pvalues.values,
    "ci_lower_95": mundlak_ci.iloc[:, 0].values,
    "ci_upper_95": mundlak_ci.iloc[:, 1].values
})

mundlak_coef.to_csv(
    STEP07_DIR / "Step_07_05_Mundlak_Coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 6. GENERIC LINEARMODELS COEFFICIENT EXTRACTOR
# ============================================================

def extract_panel_coefficients(result, model_name):

    ci = result.conf_int()

    out = pd.DataFrame({
        "model": model_name,
        "variable": result.params.index,
        "coefficient": result.params.values,
        "std_error": result.std_errors.values,
        "t_statistic": result.tstats.values,
        "p_value": result.pvalues.values,
        "ci_lower_95": ci.iloc[:, 0].values,
        "ci_upper_95": ci.iloc[:, 1].values
    })

    return out


# ============================================================
# 7. SAVE POOLED / FE / RE / TWFE COEFFICIENTS
# ============================================================

all_model_coefficients = []

if "pooled_res" in globals():
    tmp = extract_panel_coefficients(
        pooled_res,
        "Pooled OLS"
    )
    all_model_coefficients.append(tmp)

    tmp.to_csv(
        STEP07_DIR / "Step_07_06a_Pooled_OLS_Coefficients.csv",
        index=False,
        encoding="utf-8-sig"
    )


if "fe_res" in globals():
    tmp = extract_panel_coefficients(
        fe_res,
        "Country Fixed Effects"
    )
    all_model_coefficients.append(tmp)

    tmp.to_csv(
        STEP07_DIR / "Step_07_06b_Country_FE_Coefficients.csv",
        index=False,
        encoding="utf-8-sig"
    )


if "re_res" in globals():
    tmp = extract_panel_coefficients(
        re_res,
        "Random Effects"
    )
    all_model_coefficients.append(tmp)

    tmp.to_csv(
        STEP07_DIR / "Step_07_06c_Random_Effects_Coefficients.csv",
        index=False,
        encoding="utf-8-sig"
    )


if "twfe_res" in globals():
    tmp = extract_panel_coefficients(
        twfe_res,
        "Country + Year Fixed Effects"
    )
    all_model_coefficients.append(tmp)

    tmp.to_csv(
        STEP07_DIR / "Step_07_06d_TWFE_Coefficients.csv",
        index=False,
        encoding="utf-8-sig"
    )


# combined coefficients table
if len(all_model_coefficients) > 0:

    coefficients_combined = pd.concat(
        all_model_coefficients,
        ignore_index=True
    )

    coefficients_combined.to_csv(
        STEP07_DIR / "Step_07_07_All_Model_Coefficients.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 8. MODEL FIT STATISTICS
# ============================================================

model_fit_rows = []

def add_fit_stats(result, model_name):

    row = {
        "model": model_name,
        "nobs": getattr(result, "nobs", np.nan),
        "rsquared": getattr(result, "rsquared", np.nan),
        "rsquared_within": getattr(
            result,
            "rsquared_within",
            np.nan
        ),
        "rsquared_between": getattr(
            result,
            "rsquared_between",
            np.nan
        ),
        "rsquared_overall": getattr(
            result,
            "rsquared_overall",
            np.nan
        ),
        "log_likelihood": getattr(
            result,
            "loglik",
            np.nan
        )
    }

    # conventional F-statistic
    fstat = getattr(
        result,
        "f_statistic",
        None
    )

    if fstat is not None:
        row["F_statistic"] = getattr(
            fstat,
            "stat",
            np.nan
        )
        row["F_p_value"] = getattr(
            fstat,
            "pval",
            np.nan
        )

    # robust F-statistic
    fstat_r = getattr(
        result,
        "f_statistic_robust",
        None
    )

    if fstat_r is not None:
        row["Robust_F_statistic"] = getattr(
            fstat_r,
            "stat",
            np.nan
        )
        row["Robust_F_p_value"] = getattr(
            fstat_r,
            "pval",
            np.nan
        )

    model_fit_rows.append(row)


if "pooled_res" in globals():
    add_fit_stats(
        pooled_res,
        "Pooled OLS"
    )

if "fe_res" in globals():
    add_fit_stats(
        fe_res,
        "Country Fixed Effects"
    )

if "re_res" in globals():
    add_fit_stats(
        re_res,
        "Random Effects"
    )

if "twfe_res" in globals():
    add_fit_stats(
        twfe_res,
        "Country + Year Fixed Effects"
    )


model_fit = pd.DataFrame(
    model_fit_rows
)

model_fit.to_csv(
    STEP07_DIR / "Step_07_08_Model_Fit_Statistics.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. TWFE POOLABILITY TEST
# ============================================================

if "twfe_res" in globals():

    pooled_test = twfe_res.f_pooled

    poolability = pd.DataFrame({
        "test": [
            "PanelOLS F-test for poolability"
        ],
        "null_hypothesis": [
            "Entity/time effects are jointly unnecessary"
        ],
        "F_statistic": [
            getattr(
                pooled_test,
                "stat",
                np.nan
            )
        ],
        "p_value": [
            getattr(
                pooled_test,
                "pval",
                np.nan
            )
        ],
        "df_num": [
            getattr(
                pooled_test,
                "df",
                np.nan
            )
        ],
        "df_denom": [
            getattr(
                pooled_test,
                "df_denom",
                np.nan
            )
        ],
        "decision_5pct": [
            "Reject poolability"
            if getattr(
                pooled_test,
                "pval",
                1
            ) < 0.05
            else "Do not reject poolability"
        ]
    })

    poolability.to_csv(
        STEP07_DIR /
        "Step_07_09_TWFE_Poolability_Test.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 10. KEY MODEL-SELECTION RESULTS — SINGLE TABLE
# ============================================================

key_selection = pd.DataFrame({
    "test": [
        "Pooled OLS vs Country FE",
        "Mundlak FE vs RE",
        "TWFE poolability"
    ],

    "statistic": [
        float(f_test[0]),
        scalar_value(mundlak_test.fvalue),
        (
            getattr(
                twfe_res.f_pooled,
                "stat",
                np.nan
            )
            if "twfe_res" in globals()
            else np.nan
        )
    ],

    "p_value": [
        float(f_test[1]),
        scalar_value(mundlak_test.pvalue),
        (
            getattr(
                twfe_res.f_pooled,
                "pval",
                np.nan
            )
            if "twfe_res" in globals()
            else np.nan
        )
    ],

    "conclusion": [
        "Country effects are required",
        "Fixed effects preferred to random effects",
        "Pooled specification rejected"
    ]
})

key_selection.to_csv(
    STEP07_DIR /
    "Step_07_10_Model_Selection_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. CURRENT MAIN TWFE RESULT — JOURNAL-READY CSV
# ============================================================

if "twfe_res" in globals():

    ci = twfe_res.conf_int()

    twfe_main = pd.DataFrame({
        "Variable": [
            "VHCN coverage",
            "Log GDP per capita (PPS)"
        ],

        "Coefficient": [
            twfe_res.params.get(
                "vhcn_pct",
                np.nan
            ),
            twfe_res.params.get(
                "ln_gdp_pc_pps",
                np.nan
            )
        ],

        "Std. Error": [
            twfe_res.std_errors.get(
                "vhcn_pct",
                np.nan
            ),
            twfe_res.std_errors.get(
                "ln_gdp_pc_pps",
                np.nan
            )
        ],

        "t-statistic": [
            twfe_res.tstats.get(
                "vhcn_pct",
                np.nan
            ),
            twfe_res.tstats.get(
                "ln_gdp_pc_pps",
                np.nan
            )
        ],

        "p-value": [
            twfe_res.pvalues.get(
                "vhcn_pct",
                np.nan
            ),
            twfe_res.pvalues.get(
                "ln_gdp_pc_pps",
                np.nan
            )
        ],

        "95% CI Lower": [
            ci.loc[
                "vhcn_pct"
            ].iloc[0],
            ci.loc[
                "ln_gdp_pc_pps"
            ].iloc[0]
        ],

        "95% CI Upper": [
            ci.loc[
                "vhcn_pct"
            ].iloc[1],
            ci.loc[
                "ln_gdp_pc_pps"
            ].iloc[1]
        ]
    })

    twfe_main.to_csv(
        STEP07_DIR /
        "Step_07_11_Main_TWFE_Journal_Table.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 12. SAVE FULL TEXT SUMMARIES AS WELL
# ============================================================

TEXT_DIR = STEP07_DIR / "Full_Model_Summaries"
TEXT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

results_to_save = {
    "Pooled_OLS": globals().get(
        "pooled_res"
    ),
    "Country_FE": globals().get(
        "fe_res"
    ),
    "Random_Effects": globals().get(
        "re_res"
    ),
    "TWFE": globals().get(
        "twfe_res"
    )
}

for name, result in results_to_save.items():

    if result is not None:

        with open(
            TEXT_DIR /
            f"Step_07_{name}_Full_Summary.txt",
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                str(result)
            )


with open(
    TEXT_DIR /
    "Step_07_Mundlak_Full_Summary.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        mundlak.summary().as_text()
    )


# ============================================================
# 13. LIST SAVED FILES
# ============================================================

print("\nFiles saved:\n")

for p in sorted(
    STEP07_DIR.rglob("*")
):
    if p.is_file():
        print(p.name)

Output folder:
/content/drive/MyDrive/AI_Realization_Gap_EU27/05_tables/Step_07_All_Results

Files saved:

Step_07_Country_FE_Full_Summary.txt
Step_07_Mundlak_Full_Summary.txt
Step_07_Pooled_OLS_Full_Summary.txt
Step_07_Random_Effects_Full_Summary.txt
Step_07_TWFE_Full_Summary.txt
Step_07_01_Panel_Structure.csv
Step_07_02_Descriptive_Statistics.csv
Step_07_03_Pooled_vs_FE_F_Test.csv
Step_07_04_Mundlak_Test.csv
Step_07_05_Mundlak_Coefficients.csv
Step_07_06a_Pooled_OLS_Coefficients.csv
Step_07_06b_Country_FE_Coefficients.csv
Step_07_06c_Random_Effects_Coefficients.csv
Step_07_06d_TWFE_Coefficients.csv
Step_07_07_All_Model_Coefficients.csv
Step_07_08_Model_Fit_Statistics.csv
Step_07_09_TWFE_Poolability_Test.csv
Step_07_10_Model_Selection_Summary.csv
Step_07_11_Main_TWFE_Journal_Table.csv


In [ ]:
# All Step 07 results are saved after diagnostics (cells below).
# Do not create a preliminary ZIP before diagnostics are generated.


In [ ]:
# ZIP creation and download occur after all diagnostics in the final cells.
# This cell is intentionally deferred to prevent downloading an incomplete ZIP.


In [ ]:
# ============================================================
# STEP 07 DIAGNOSTICS — PREPARE RESIDUAL DATA
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import norm, chi2
import statsmodels.formula.api as smf

DIAG_DIR = STEP07_DIR

df_diag = df.sort_values(
    ["iso3", "year"]
).copy()

# TWFE model for residual diagnostics
twfe_diag = smf.ols(
    """
    ai_realization_gap ~
    vhcn_pct +
    ln_gdp_pc_pps +
    C(iso3) +
    C(year)
    """,
    data=df_diag
).fit()

df_diag["twfe_resid"] = twfe_diag.resid

print("Residual observations:", df_diag["twfe_resid"].notna().sum())

Residual observations: 108


In [ ]:
# ============================================================
# HETEROSKEDASTICITY DIAGNOSTIC
# ============================================================

country_resid_var = (
    df_diag
    .groupby("iso3")["twfe_resid"]
    .agg(
        n="count",
        residual_mean="mean",
        residual_variance="var",
        residual_std="std"
    )
    .reset_index()
)

display(country_resid_var)

country_resid_var.to_csv(
    DIAG_DIR /
    "Step_07_12_Residual_Variance_by_Country.csv",
    index=False,
    encoding="utf-8-sig"
)

,iso3,n,residual_mean,residual_variance,residual_std
0,AUT,4,1.051936e-14,0.003668,0.060566
1,BEL,4,6.640521e-15,0.017258,0.131372
2,BGR,4,8.409939e-15,0.004918,0.070129
3,CYP,4,1.189326e-14,0.002898,0.053830
4,CZE,4,1.743744e-14,0.000908,0.030132
5,DEU,4,1.044824e-14,0.000476,0.021827
6,DNK,4,5.495604e-15,0.033737,0.183676
7,ESP,4,5.419276e-15,0.002777,0.052700
8,EST,4,9.769963e-15,0.003161,0.056219
9,FIN,4,9.499346e-15,0.012155,0.110248


In [ ]:
# ============================================================
# BROWN-FORSYTHE / LEVENE TEST
# ============================================================

from scipy.stats import levene

groups = [
    g["twfe_resid"].dropna().values
    for _, g in df_diag.groupby("iso3")
]

lev_stat, lev_p = levene(
    *groups,
    center="median"
)

levene_result = pd.DataFrame({
    "test": [
        "Brown-Forsythe / Levene test"
    ],
    "null_hypothesis": [
        "Residual variances are equal across countries"
    ],
    "statistic": [
        lev_stat
    ],
    "p_value": [
        lev_p
    ],
    "decision_5pct": [
        "Reject homoskedasticity"
        if lev_p < 0.05
        else "Do not reject homoskedasticity"
    ]
})

display(levene_result)

levene_result.to_csv(
    DIAG_DIR /
    "Step_07_13_Heteroskedasticity_Test.csv",
    index=False,
    encoding="utf-8-sig"
)

,test,null_hypothesis,statistic,p_value,decision_5pct
0,Brown-Forsythe / Levene test,Residual variances are equal across countries,1.532891,0.075806,Do not reject homoskedasticity


In [ ]:
# ============================================================
# SERIAL CORRELATION DIAGNOSTIC
# ============================================================

serial_df = df_diag.copy()

serial_df["resid_lag1"] = (
    serial_df
    .groupby("iso3")["twfe_resid"]
    .shift(1)
)

serial_sample = serial_df.dropna(
    subset=[
        "twfe_resid",
        "resid_lag1"
    ]
).copy()

serial_model = smf.ols(
    "twfe_resid ~ resid_lag1",
    data=serial_sample
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": serial_sample["iso3"]
    }
)

serial_ci = serial_model.conf_int()

serial_result = pd.DataFrame({
    "test": [
        "Residual AR(1) diagnostic"
    ],
    "coefficient_lag1": [
        serial_model.params[
            "resid_lag1"
        ]
    ],
    "std_error": [
        serial_model.bse[
            "resid_lag1"
        ]
    ],
    "test_statistic": [
        serial_model.tvalues[
            "resid_lag1"
        ]
    ],
    "p_value": [
        serial_model.pvalues[
            "resid_lag1"
        ]
    ],
    "ci_lower_95": [
        serial_ci.loc[
            "resid_lag1"
        ].iloc[0]
    ],
    "ci_upper_95": [
        serial_ci.loc[
            "resid_lag1"
        ].iloc[1]
    ],
    "interpretation_5pct": [
        "Evidence of residual serial correlation"
        if serial_model.pvalues["resid_lag1"] < 0.05
        else "No statistically significant residual serial correlation"
    ]
})

display(serial_result)

serial_result.to_csv(
    DIAG_DIR /
    "Step_07_14_Serial_Correlation_Diagnostic.csv",
    index=False,
    encoding="utf-8-sig"
)

,test,coefficient_lag1,std_error,test_statistic,p_value,ci_lower_95,ci_upper_95,interpretation_5pct
0,Residual AR(1) diagnostic,-0.064956,0.072105,-0.900852,0.367667,-0.206279,0.076367,No statistically significant residual serial c...


In [ ]:
# ============================================================
# PESARAN CD TEST ON TWFE RESIDUALS
# ============================================================

resid_panel = (
    df_diag
    .pivot(
        index="year",
        columns="iso3",
        values="twfe_resid"
    )
)

corr_matrix = resid_panel.corr()

N_cd = corr_matrix.shape[0]
T_cd = resid_panel.shape[0]

pairwise_corrs = []

for i in range(N_cd):
    for j in range(i + 1, N_cd):
        rho = corr_matrix.iloc[i, j]

        if np.isfinite(rho):
            pairwise_corrs.append(rho)

pairwise_corrs = np.array(
    pairwise_corrs
)

CD_stat = np.sqrt(
    2 * T_cd /
    (N_cd * (N_cd - 1))
) * pairwise_corrs.sum()

CD_p = 2 * (
    1 - norm.cdf(
        abs(CD_stat)
    )
)

mean_pair_corr = (
    pairwise_corrs.mean()
)

pesaran_cd = pd.DataFrame({
    "test": [
        "Pesaran CD test on TWFE residuals"
    ],
    "N": [
        N_cd
    ],
    "T": [
        T_cd
    ],
    "CD_statistic": [
        CD_stat
    ],
    "p_value": [
        CD_p
    ],
    "mean_pairwise_residual_correlation": [
        mean_pair_corr
    ],
    "decision_5pct": [
        "Reject cross-sectional independence"
        if CD_p < 0.05
        else "Do not reject cross-sectional independence"
    ]
})

display(pesaran_cd)

pesaran_cd.to_csv(
    DIAG_DIR /
    "Step_07_15_Pesaran_CD_Residuals.csv",
    index=False,
    encoding="utf-8-sig"
)

,test,N,T,CD_statistic,p_value,mean_pairwise_residual_correlation,decision_5pct
0,Pesaran CD test on TWFE residuals,27,4,-1.18667,0.235358,-0.03167,Do not reject cross-sectional independence


In [ ]:
# ============================================================
# PAIRWISE RESIDUAL CORRELATIONS
# ============================================================

pair_rows = []

countries = corr_matrix.columns.tolist()

for i in range(len(countries)):
    for j in range(i + 1, len(countries)):

        pair_rows.append({
            "country_i": countries[i],
            "country_j": countries[j],
            "residual_correlation":
                corr_matrix.iloc[i, j]
        })

pairwise_resid_corr = pd.DataFrame(
    pair_rows
)

pairwise_resid_corr.to_csv(
    DIAG_DIR /
    "Step_07_16_Pairwise_Residual_Correlations.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    pairwise_resid_corr
    .sort_values(
        "residual_correlation",
        ascending=False
    )
    .head(10)
)

,country_i,country_j,residual_correlation
180,EST,FIN,0.993752
19,AUT,NLD,0.988591
341,PRT,ROU,0.988215
317,LVA,POL,0.976043
222,FRA,LVA,0.974265
268,HUN,PRT,0.970460
61,BGR,HUN,0.963799
177,ESP,SVK,0.962954
246,HRV,HUN,0.950886
284,IRL,SWE,0.938978


In [ ]:
# ============================================================
# DIAGNOSTIC SUMMARY TABLE
# ============================================================

diagnostic_summary = pd.DataFrame({
    "diagnostic": [
        "Heteroskedasticity",
        "Serial correlation",
        "Cross-sectional dependence"
    ],

    "test": [
        "Brown-Forsythe / Levene",
        "Residual AR(1) diagnostic",
        "Pesaran CD"
    ],

    "statistic": [
        lev_stat,
        serial_model.tvalues[
            "resid_lag1"
        ],
        CD_stat
    ],

    "p_value": [
        lev_p,
        serial_model.pvalues[
            "resid_lag1"
        ],
        CD_p
    ],

    "decision_5pct": [
        (
            "Detected"
            if lev_p < 0.05
            else "Not detected"
        ),

        (
            "Detected"
            if serial_model.pvalues[
                "resid_lag1"
            ] < 0.05
            else "Not detected"
        ),

        (
            "Detected"
            if CD_p < 0.05
            else "Not detected"
        )
    ],

    "note": [
        "Country-level residual variance diagnostic",
        "T=4; interpret cautiously",
        "N=27, T=4; low-T diagnostic"
    ]
})

display(diagnostic_summary)

diagnostic_summary.to_csv(
    DIAG_DIR /
    "Step_07_17_Diagnostic_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

,diagnostic,test,statistic,p_value,decision_5pct,note
0,Heteroskedasticity,Brown-Forsythe / Levene,1.532891,0.075806,Not detected,Country-level residual variance diagnostic
1,Serial correlation,Residual AR(1) diagnostic,-0.900852,0.367667,Not detected,T=4; interpret cautiously
2,Cross-sectional dependence,Pesaran CD,-1.186670,0.235358,Not detected,"N=27, T=4; low-T diagnostic"


In [ ]:
# ============================================================
# SAVE TWFE RESIDUAL DATA
# ============================================================

residual_output = df_diag[
    [
        "geo",
        "country",
        "iso3",
        "year",
        "ai_realization_gap",
        "vhcn_pct",
        "ln_gdp_pc_pps",
        "twfe_resid"
    ]
].copy()

residual_output.to_csv(
    DIAG_DIR /
    "Step_07_18_TWFE_Residuals.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP 07 — ROBUSTNESS CHECKS (MANUSCRIPT TABLE 5)
# Original econometric models and diagnostics remain unchanged.
# ============================================================
from linearmodels.panel import PanelOLS
import numpy as np
import pandas as pd

robustness_specs = [
    ("Baseline TWFE", "ai_realization_gap", ["vhcn_pct", "ln_gdp_pc_pps"]),
    ("Z-gap", "ai_realization_gap_z", ["vhcn_pct", "ln_gdp_pc_pps"]),
    ("EW-gap", "ai_realization_gap_equal", ["vhcn_pct", "ln_gdp_pc_pps"]),
    ("Cloud-augmented", "ai_realization_gap", ["vhcn_pct", "ln_gdp_pc_pps", "cloud_pct"]),
]

robustness_models = {}
robustness_coef_rows = []
robustness_fit_rows = []
robustness_sample_rows = []
for spec_name, dependent, regressors in robustness_specs:
    required_cols = ["iso3", "year", dependent] + regressors
    if not set(required_cols).issubset(df.columns):
        raise KeyError(f"Missing variables for {spec_name}: {set(required_cols)-set(df.columns)}")
    sample = df[required_cols].dropna().copy()
    if sample.duplicated(["iso3", "year"]).any():
        raise ValueError(f"Duplicate country-years in {spec_name}")
    sample = sample.set_index(["iso3", "year"]).sort_index()
    if spec_name != "Cloud-augmented" and len(sample) != 108:
        raise ValueError(f"{spec_name} should contain 108 observations; found {len(sample)}")
    if spec_name == "Cloud-augmented" and len(sample) != 85:
        raise ValueError(f"Cloud subsample expected 85 observations; found {len(sample)}")
    model = PanelOLS(sample[dependent], sample[regressors],
                     entity_effects=True, time_effects=True, drop_absorbed=True)
    result = model.fit(cov_type="clustered", cluster_entity=True)
    robustness_models[spec_name] = result
    for variable in result.params.index:
        robustness_coef_rows.append({
            "model": spec_name, "dependent": dependent, "variable": variable,
            "coefficient": float(result.params[variable]),
            "std_error": float(result.std_errors[variable]),
            "t_statistic": float(result.tstats[variable]),
            "p_value": float(result.pvalues[variable]),
            "ci_lower_95": float(result.conf_int().loc[variable].iloc[0]),
            "ci_upper_95": float(result.conf_int().loc[variable].iloc[1]),
            "nobs": int(result.nobs),
        })
    robustness_fit_rows.append({
        "model": spec_name, "dependent": dependent,
        "nobs": int(result.nobs), "n_countries": int(sample.index.get_level_values(0).nunique()),
        "n_years": int(sample.index.get_level_values(1).nunique()),
        "r2_within_model": float(result.rsquared),
        "f_robust_stat": float(result.f_statistic_robust.stat),
        "f_robust_p": float(result.f_statistic_robust.pval),
        "cluster": "country", "country_fe": True, "year_fe": True
    })
    robustness_sample_rows += [{"model": spec_name, "iso3": iso3, "year": int(year)}
                               for iso3, year in sample.index]

robustness_coefs = pd.DataFrame(robustness_coef_rows)
robustness_fit = pd.DataFrame(robustness_fit_rows)
robustness_sample = pd.DataFrame(robustness_sample_rows)
robustness_coefs.to_csv(STEP07_DIR / "Step_07_19_Robustness_Coefficients.csv", index=False, encoding="utf-8-sig")
robustness_fit.to_csv(STEP07_DIR / "Step_07_20_Robustness_Model_Fit.csv", index=False, encoding="utf-8-sig")
robustness_sample.to_csv(STEP07_DIR / "Step_07_21_Robustness_Samples.csv", index=False, encoding="utf-8-sig")

# Paper Table 5: values are reference entries, NOT imposed as estimation results.
# Baseline standard error differs between manuscript Table 4 and Table 5.
paper_reference = {
    "Baseline TWFE": {"vhcn_pct": (0.0012, 0.0018), "ln_gdp_pc_pps": (1.4279, 0.3785)},
    "Z-gap": {"vhcn_pct": (0.0052, 0.0079), "ln_gdp_pc_pps": (6.5810, 1.6394)},
    "EW-gap": {"vhcn_pct": (0.0013, 0.0018), "ln_gdp_pc_pps": (1.4343, 0.3744)},
    "Cloud-augmented": {"vhcn_pct": (0.0021, 0.0021),
                        "ln_gdp_pc_pps": (1.5098, 0.4704), "cloud_pct": (0.0047, 0.0025)}
}
check_rows = []
for row in robustness_coef_rows:
    target = paper_reference[row["model"]].get(row["variable"])
    if target is None: continue
    paper_coef, paper_se = target
    check_rows.append({
        "model": row["model"], "variable": row["variable"],
        "computed_coefficient": row["coefficient"], "paper_coefficient": paper_coef,
        "coefficient_diff": row["coefficient"] - paper_coef,
        "computed_std_error": row["std_error"], "paper_std_error": paper_se,
        "std_error_diff": row["std_error"] - paper_se,
        "coefficient_matches_4dp": round(row["coefficient"], 4) == paper_coef,
        "se_matches_4dp": round(row["std_error"], 4) == paper_se,
    })
robustness_validation = pd.DataFrame(check_rows)
robustness_validation.to_csv(STEP07_DIR / "Step_07_22_Manuscript_Table5_Validation.csv",
                             index=False, encoding="utf-8-sig")

# Journal-friendly manuscript Table 5 reconstructed FROM model estimates.
journal_rows = []
for variable in ["vhcn_pct", "ln_gdp_pc_pps", "cloud_pct"]:
    r = {"variable": variable}
    for spec_name, _, _ in robustness_specs:
        sub = robustness_coefs[(robustness_coefs.model == spec_name) &
                               (robustness_coefs.variable == variable)]
        if len(sub):
            v = sub.iloc[0]
            stars = "***" if v.p_value < .01 else "**" if v.p_value < .05 else "*" if v.p_value < .10 else ""
            r[spec_name] = f"{v.coefficient:.4f}{stars} ({v.std_error:.4f})"
        else: r[spec_name] = "—"
    journal_rows.append(r)
nrow = {"variable": "Observations"}
for item in robustness_fit_rows: nrow[item["model"]] = item["nobs"]
journal_rows.append(nrow)
robustness_journal = pd.DataFrame(journal_rows)
robustness_journal.to_csv(STEP07_DIR / "Step_07_23_Robustness_Journal_Table.csv",
                          index=False, encoding="utf-8-sig")

for spec_name, result in robustness_models.items():
    (STEP07_DIR / f"Step_07_Robustness_{spec_name.replace(' ', '_').replace('-', '_')}_Summary.txt").write_text(
        str(result.summary), encoding="utf-8"
    )
print("Robustness models saved:", len(robustness_models))
print(robustness_journal.to_string(index=False))
print("\nManuscript Table 5 mismatches (4 decimal places):")
print(robustness_validation.loc[~(robustness_validation.coefficient_matches_4dp &
                                    robustness_validation.se_matches_4dp)].to_string(index=False))


In [ ]:
# ============================================================
# FINAL ZIP — require diagnostics and robustness outputs
# ============================================================
import zipfile

required_csv = [
    "Step_07_01_Panel_Structure.csv",
    "Step_07_02_Descriptive_Statistics.csv",
    "Step_07_03_Pooled_vs_FE_F_Test.csv",
    "Step_07_04_Mundlak_Test.csv",
    "Step_07_05_Mundlak_Coefficients.csv",
    "Step_07_06a_Pooled_OLS_Coefficients.csv",
    "Step_07_06b_Country_FE_Coefficients.csv",
    "Step_07_06c_Random_Effects_Coefficients.csv",
    "Step_07_06d_TWFE_Coefficients.csv",
    "Step_07_07_All_Model_Coefficients.csv",
    "Step_07_08_Model_Fit_Statistics.csv",
    "Step_07_09_TWFE_Poolability_Test.csv",
    "Step_07_10_Model_Selection_Summary.csv",
    "Step_07_11_Main_TWFE_Journal_Table.csv",
    "Step_07_12_Residual_Variance_by_Country.csv",
    "Step_07_13_Heteroskedasticity_Test.csv",
    "Step_07_14_Serial_Correlation_Diagnostic.csv",
    "Step_07_15_Pesaran_CD_Residuals.csv",
    "Step_07_16_Pairwise_Residual_Correlations.csv",
    "Step_07_17_Diagnostic_Summary.csv",
    "Step_07_18_TWFE_Residuals.csv",
    "Step_07_19_Robustness_Coefficients.csv",
    "Step_07_20_Robustness_Model_Fit.csv",
    "Step_07_21_Robustness_Samples.csv",
    "Step_07_22_Manuscript_Table5_Validation.csv",
    "Step_07_23_Robustness_Journal_Table.csv",
]
required_txt = [
    "Step_07_Pooled_OLS_Full_Summary.txt",
    "Step_07_Country_FE_Full_Summary.txt",
    "Step_07_Random_Effects_Full_Summary.txt",
    "Step_07_TWFE_Full_Summary.txt",
    "Step_07_Mundlak_Full_Summary.txt",
]
robustness_txt = [STEP07_DIR / f"Step_07_Robustness_{name.replace(' ', '_').replace('-', '_')}_Summary.txt" for name in robustness_models]
required_paths = ([STEP07_DIR / name for name in required_csv] +
                  [STEP07_DIR / "Full_Model_Summaries" / name for name in required_txt] + robustness_txt)
missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing Step 07 results:\n" + "\n".join(missing))

ZIP_PATH = PROJECT_ROOT / "Step_07_All_Econometric_Results.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in required_paths:
        zf.write(path, arcname=str(path.relative_to(STEP07_DIR)))
with zipfile.ZipFile(ZIP_PATH) as zf:
    if zf.testzip() is not None:
        raise IOError("ZIP integrity check failed")
print(f"ZIP ready with {len(required_paths)} verified files: {ZIP_PATH}")


In [ ]:
from google.colab import files

files.download(
    str(ZIP_PATH)
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>